# Experiment 4（前段）: ベースモデルの Known / Unknown プローブ

**fine-tuning の前に**、評価 QA（`make_qa.py` の出力）をベースモデルに closed-book で解かせ、各事実を「すでに知っている知識か」で分類する。
分類は Gekhman et al. 2024（SliCK）に従う。**英語・日本語の両方で Unknown の事実だけ** を注入対象にする。

SliCK は連続量 `PCorrect(q, a; M, T)` を推定してから離散化する。`N_EX` 通りの few-shot プロンプト × 言い回し（`make_qa.py` は 2 本）のそれぞれで greedy 1 回（T=0）と `N_SAMPLES` 回のサンプル（T=0.5）を取り、

- `PCorrect(T=0)` = greedy の正解率（`N_EX × 2` 回中）
- `PCorrect(T>0)` = サンプルの正解率（`N_EX × 2 × N_SAMPLES` 回中）

| ラベル | 条件 |
|---|---|
| HighlyKnown | `PCorrect(T=0) == 1` |
| MaybeKnown | `0 < PCorrect(T=0) < 1` |
| WeaklyKnown | `PCorrect(T=0) == 0` かつ `PCorrect(T>0) > 0` |
| Unknown | `PCorrect(T>=0) == 0` |

**論文からの意図的な逸脱2点**（`scripts/probe_known.py` の docstring にも記載）:
1. 論文は「q と同じ relation の exemplar」を Nex 通りランダムに選んでプロンプトの分散を作るが、ここでそれをやると**注入対象の post-cutoff 事実が exemplar に漏れて closed-book が壊れる**。代わりに `pk.FEWSHOT_SETS`（カットオフ前の事実だけで作った 4 セット）× 言い回し 2 本で分散を作る。
2. 論文の採点は exact match。ここでは答えが自由記述で長い（"American independent film distribution company"）ので、answer + 別名に対する正規化部分一致で採点する。短い ASCII の答え（"17", "AI"）はトークン境界一致を要求する。参考として word-level F1 の最大値（`f1_greedy_max` / `f1_sample_max`）も記録するので、**再実行せずに Unknown の閾値を後から見直せる**。

**なぜ必要か**: ベースが日本語で答えられる事実を注入すると、fine-tuning 後に日本語で正解しても「転移」なのか「事前知識」なのか区別できない。
名前から推測できる事実（"HydroGNSS の開発元は ESA"）もここで Known 側に落ちるので、手で除外しなくてよい。
ついでに、ベースモデルの **言語別の棄権率と誤答率** を測る（棄権しやすさの言語差は交絡になるので、先に数字を持っておく）。

**入力**: `experiment_data/eval_qa.jsonl`（EN 2 言い回し + JA 2 言い回し + 別名）と `experiment_data/facts.jsonl`
**出力**: `experiment_data/facts_probed.jsonl`（事実行 + `known_en` / `known_ja` + `PCorrect` + 生の回答）— `scripts/probe_known.py` と同じ形式なので、後続の `make_paraphrases.py --unknown-only` と `experiment_4_finetune.ipynb` はそのまま使える。`experiment_data/probe_summary.json` に集計も残す。

**`scripts/probe_known.py` との関係**: プロンプト・判定・分類の関数はそのスクリプトから import して使う（同じ挙動）。このノートブックは、判定器の感度確認（コントロールセット）、結果の検査（どの事実が Known になったか、記事ごとの既知率、日本語回答の文字種）を加えたもの。

**所要時間**: 1 事実あたり 2 言語 × `N_EX × 2 × (1 + N_SAMPLES)` 生成。既定（`N_EX=4, N_SAMPLES=8`）で **144 生成/事実** → 全 4,074 事実は **クラスタ GPU 前提**（MPS だと数十時間なので回さない）。ローカルの試運転は `N_EX=1, N_SAMPLES=3`（16 生成/事実）。言語ごとに生成結果をキャッシュするので、途中で落ちても再実行で続きから。まず `LIMIT = 20` で試す。

**実行順**: `extract_facts.py` → `make_qa.py` → **このノートブック** → `make_paraphrases.py --unknown-only` → `experiment_4_finetune.ipynb`

In [ ]:
# ===== Config =====  papermill の "parameters" セル
model_name = "Qwen/Qwen2.5-1.5B"   # ← プローブは fine-tuning に使うベースモデルと同じものにする

QA_FILE = "experiment_data/eval_qa.jsonl"
FACTS_FILE = "experiment_data/facts.jsonl"
OUT_FILE = "experiment_data/facts_probed.jsonl"
SUMMARY_FILE = "experiment_data/probe_summary.json"
CACHE_DIR = "experiment_data/probe_cache"   # 言語ごとの生成結果（再開用）。model / 言語 / QA 内容のハッシュで別ファイル

N_EX = 4               # few-shot プロンプトの種類数（SliCK の Nex）。pk.FEWSHOT_SETS の数が上限
N_SAMPLES = 8          # プロンプトごとの temperature サンプル数（WeaklyKnown の検出力を決める）
TEMPERATURE = 0.5
TOP_P = 1.0            # 論文は T=0.5 のみ指定。裾を切ると WeaklyKnown が減るので既定は切らない
BATCH = 32             # MPS でメモリが苦しければ 16
LIMIT = 0              # 0 = 全事実。試し: 20
DEVICE = "auto"        # "auto" | "cuda" | "mps" | "cpu"
RUN_CONTROL = True     # 既知 / 架空の事実のコントロールセットで判定器の感度を先に確認する

In [ ]:
import collections, hashlib, json, os, sys, time
from pathlib import Path

import pandas as pd
import torch

ROOT = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
assert (ROOT / "scripts" / "probe_known.py").exists(), f"project root not found from {Path.cwd()}"
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "scripts"))
import probe_known as pk   # FEWSHOT_SETS / is_correct / classify / max_f1 / probe_language / Generator — スクリプトと同じ挙動

device = DEVICE if DEVICE != "auto" else ("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model_slug = model_name.rstrip("/").split("/")[-1]
CACHE = ROOT / CACHE_DIR
CACHE.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 60, "display.width", 200)

def read_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

N_GREEDY = N_EX * 2    # 事実・言語あたりの greedy 回数（= PCorrect(T=0) の分母）
assert N_EX <= len(pk.FEWSHOT_SETS["en"]), f"N_EX={N_EX} > few-shot セット数 {len(pk.FEWSHOT_SETS['en'])}"
print(f"device {device} | model {model_name} | nex {N_EX} × 2 phrasings = {N_GREEDY} greedy "
      f"+ {N_GREEDY * N_SAMPLES} samples @ T={TEMPERATURE} top_p={TOP_P} | batch {BATCH}")

## 1. モデル読み込み

In [ ]:
t0 = time.time()
gen = pk.Generator(model_name, device)
print(f"loaded in {time.time() - t0:.0f}s")

def probe(questions_per_fact, golds_list, lang):
    """questions_per_fact: 事実ごとの言い回しリスト（make_qa.py は 2 本、コントロールセットは 1 本）。
    戻り値: 事実ごとの {label, p_correct_greedy, p_correct_sample, f1_greedy_max, f1_sample_max,
    n_greedy, n_sample, greedy, samples, abstain_greedy, wrong_greedy}。
    中身は scripts/probe_known.py の probe_language そのままなので、スクリプトと結果が一致する。"""
    return pk.probe_language(gen, questions_per_fact, golds_list, lang, nex=N_EX, samples=N_SAMPLES,
                             temperature=TEMPERATURE, top_p=TOP_P, batch=BATCH)

## 2. コントロールセット（判定器の感度確認）

既知の事実が Known に、架空 / カットオフ後の事実が Unknown になることを先に確認する。
ここで既知の事実が Unknown になるなら、プロンプト形式か判定（`is_correct` の別名）に問題があり、本番の Unknown ラベルを信用できない。

In [ ]:
CONTROL = pk.CONTROL   # 既知6件 + 架空1件 + post-cutoff 1件。定義は scripts/probe_known.py（scripts/probe_smoke.py と共通）

if RUN_CONTROL:
    t0 = time.time()
    ctrl = {}
    for lang, qi, gi in (("en", 0, 2), ("ja", 1, 3)):
        # 言い回しは 1 本。プロンプトの分散は N_EX の few-shot セットが作るので greedy は N_EX 回
        ctrl[lang] = probe([[c[qi]] for c in CONTROL], [c[gi] for c in CONTROL], lang)
    ctrl_time = time.time() - t0
    ctrl_df = pd.DataFrame([{"question": c[0][:45], "expected": c[4],
                             "en": ctrl["en"][i]["label"], "en_pc": ctrl["en"][i]["p_correct_greedy"], "en_answer": ctrl["en"][i]["greedy"][0],
                             "ja": ctrl["ja"][i]["label"], "ja_pc": ctrl["ja"][i]["p_correct_greedy"], "ja_answer": ctrl["ja"][i]["greedy"][0]} for i, c in enumerate(CONTROL)])
    print(f"control set: {ctrl_time:.0f}s for {len(CONTROL)} questions × 2 langs")
    display(ctrl_df)

## 3. 評価 QA と事実表の読み込み

In [ ]:
assert Path(QA_FILE).exists(), f"{QA_FILE} がない。先に scripts/make_qa.py を実行する"
facts = {f["fact_id"]: f for f in read_jsonl(FACTS_FILE)}
qa_all = read_jsonl(QA_FILE)
qa = [q for q in qa_all if q["fact_id"] in facts and len(q.get("q_en", [])) >= 2 and len(q.get("q_ja", [])) >= 2]
if LIMIT:
    qa = qa[:LIMIT]
no_qa = len(facts) - len({q["fact_id"] for q in qa_all})
print(f"facts {len(facts)} | eval QA {len(qa_all)} (facts without QA: {no_qa}) | probing {len(qa)} facts "
      f"= {len(qa) * 2 * N_GREEDY * (1 + N_SAMPLES)} generations")
if RUN_CONTROL:
    est = ctrl_time / (len(CONTROL) * 2) * (len(qa) * 2) * 2   # 本番は言い回しが 2 本なので ×2
    print(f"概算所要時間（コントロールセットの速度から）: {est / 60:.0f} 分")
pd.DataFrame(qa[:3])[["fact_id", "article", "type", "q_en", "q_ja", "answer", "aliases_ja"]]

## 4. プローブ本体

言語ごとに greedy（`N_EX` セット × 2 言い回し）→ temperature サンプル（その各プロンプトから `N_SAMPLES` 回）。生成結果は `CACHE_DIR` に保存し、同じ model・言語・QA 内容・設定なら再実行時にスキップする。

In [ ]:
def qa_hash(key):
    h = hashlib.md5("\n".join(q["fact_id"] + "\t" + "\t".join(q[key][:2]) for q in qa).encode("utf-8")).hexdigest()[:10]
    return h

results = {q["fact_id"]: {} for q in qa}
t0 = time.time()
for lang, key in (("en", "q_en"), ("ja", "q_ja")):
    cache = CACHE / f"{model_slug}-{lang}-{len(qa)}-{qa_hash(key)}-x{N_EX}s{N_SAMPLES}t{TEMPERATURE}p{TOP_P}.json"
    golds_list = [[q["answer"]] + q.get("aliases_en", []) + (q.get("aliases_ja", []) if lang == "ja" else []) for q in qa]
    if cache.exists():
        rows = json.load(open(cache, encoding="utf-8"))
        print(f"[{lang}] cached ← {cache.name}")
    else:
        print(f"[{lang}] probing {len(qa)} facts ...")
        rows = probe([q[key][:2] for q in qa], golds_list, lang)
        json.dump(rows, open(cache, "w", encoding="utf-8"), ensure_ascii=False)
        print(f"[{lang}] done in {(time.time() - t0) / 60:.1f} min → {cache.name}")
    for q, r in zip(qa, rows):
        results[q["fact_id"]][lang] = r
print(f"total {(time.time() - t0) / 60:.1f} min")

## 5. 集計

- ラベル分布（言語別）と EN × JA のクロス表 → **両言語 Unknown** が注入セット
- greedy の棄権率・誤答率（言い回し単位）: ベースモデルの言語差。誤答 = 正解でも棄権でもない（未知の事実に自信ありげに答えている）
- 事実型・ドメイン別の Unknown 率（Zhao et al. の分析軸）

In [ ]:
LABELS = ["HighlyKnown", "MaybeKnown", "WeaklyKnown", "Unknown"]
df = pd.DataFrame([{
    "fact_id": q["fact_id"], "article": q["article"], "type": q["type"],
    "domain": (facts[q["fact_id"]].get("domains") or ["?"])[0],
    "subject": facts[q["fact_id"]]["subject"], "relation": facts[q["fact_id"]]["relation"], "object": facts[q["fact_id"]]["object"],
    "known_en": results[q["fact_id"]]["en"]["label"], "known_ja": results[q["fact_id"]]["ja"]["label"],
    "pc_en": results[q["fact_id"]]["en"]["p_correct_greedy"], "pc_ja": results[q["fact_id"]]["ja"]["p_correct_greedy"],
    "f1_en": results[q["fact_id"]]["en"]["f1_greedy_max"], "f1_ja": results[q["fact_id"]]["ja"]["f1_greedy_max"],
    "greedy_en": results[q["fact_id"]]["en"]["greedy"], "greedy_ja": results[q["fact_id"]]["ja"]["greedy"],
    "abstain_en": results[q["fact_id"]]["en"]["abstain_greedy"], "abstain_ja": results[q["fact_id"]]["ja"]["abstain_greedy"],
    "wrong_en": results[q["fact_id"]]["en"]["wrong_greedy"], "wrong_ja": results[q["fact_id"]]["ja"]["wrong_greedy"],
} for q in qa])
df["inject"] = (df["known_en"] == "Unknown") & (df["known_ja"] == "Unknown")

dist = pd.DataFrame({lang: df[f"known_{lang}"].value_counts().reindex(LABELS, fill_value=0) for lang in ("en", "ja")})
dist["en %"] = (100 * dist["en"] / len(df)).round(1); dist["ja %"] = (100 * dist["ja"] / len(df)).round(1)
cross = pd.crosstab(df["known_en"], df["known_ja"]).reindex(index=LABELS, columns=LABELS, fill_value=0)
rates = pd.DataFrame({lang: {"abstain (greedy)": df[f"abstain_{lang}"].sum() / (N_GREEDY * len(df)),
                             "wrong (greedy)": df[f"wrong_{lang}"].sum() / (N_GREEDY * len(df))} for lang in ("en", "ja")}).T
print(f"probed facts: {len(df)}   injection set (Unknown in both languages): {int(df['inject'].sum())} ({100 * df['inject'].mean():.0f}%)\n")
print("label distribution:\n" + dist.to_string())
print("\nEN (rows) × JA (cols):\n" + cross.to_string())
print("\nbase-model greedy rates per prompt:\n" + rates.round(3).to_string())
near = df[df["inject"] & (df[["f1_en", "f1_ja"]].max(axis=1) >= 0.8)]
print(f"\n注入対象のうち word-F1 >= 0.8 の惜しい誤判定候補: {len(near)} 件"
      f"（部分一致では不正解だが実質当てている可能性。categorical 型に出やすい）")

In [ ]:
# 事実型・ドメイン別: Unknown の割合と注入セットの割合
by_type = df.groupby("type").agg(n=("fact_id", "size"), unknown_en=("known_en", lambda s: (s == "Unknown").mean()),
                                 unknown_ja=("known_ja", lambda s: (s == "Unknown").mean()), inject=("inject", "mean")).round(3)
by_domain = df.groupby("domain").agg(n=("fact_id", "size"), unknown_en=("known_en", lambda s: (s == "Unknown").mean()),
                                     unknown_ja=("known_ja", lambda s: (s == "Unknown").mean()), inject=("inject", "mean")).round(3)
print("by fact type:\n" + by_type.to_string())
print("\nby domain:\n" + by_domain.to_string())

## 6. 検査

1. **記事ごとの既知率**: 事実の大半が Known の記事は、記事は 2025 年作成でも実体自体はカットオフ前から存在する可能性が高い（例: 改名・再編された組織）。注入セットからは自動的に外れるが、データの素性として把握しておく
2. **EN で HighlyKnown になった事実**: 名前から推測できる事実や、一般常識に近い事実。どんなものが落ちたかを見る
3. **Unknown だが棄権せず答えた例**: ベースモデルの confabulation の実例（仮説の前提）
4. **日本語回答の文字種**: 日本語の質問に中国語（簡体字）で答えることがある。かな無し・漢字のみの回答を一覧にして、`aliases_ja` に足すべき表記が無いか目視する

### 6.0 各 QA の生の回答を見る

`qa_view` が全事実 × 2 言語の一覧。`show(fact_id)` で 1 事実を全部（質問 2 通り、greedy 全部、サンプルを出現回数つきで集約、✓ が正解判定）表示する。

```python
qa_view[qa_view.label == "Unknown"].head(30)            # 注入候補になった事実
qa_view[(qa_view.lang == "ja") & (qa_view.pc_greedy > 0)]  # 日本語で答えられた事実
show("83148603-2")                                       # 1 件を詳しく
```


In [ ]:
# 全事実 × 2言語の一覧。label / PCorrect / 質問 / 正解 / greedy 回答 / よく出たサンプル
qa_view = pd.DataFrame([{
    "fact_id": q["fact_id"], "type": q["type"], "article": q["article"], "lang": lang,
    "label": results[q["fact_id"]][lang]["label"],
    "pc_greedy": results[q["fact_id"]][lang]["p_correct_greedy"],
    "pc_sample": results[q["fact_id"]][lang]["p_correct_sample"],
    "question": q[key][0],
    "gold": q["answer"],
    "greedy": results[q["fact_id"]][lang]["greedy"],
    "samples_top": [a for a, _ in collections.Counter(results[q["fact_id"]][lang]["samples"]).most_common(4)],
} for q in qa for lang, key in (("en", "q_en"), ("ja", "q_ja"))])

print(f"{len(qa_view)} rows（{len(qa)} 事実 × 2 言語）")
display(qa_view.head(20))


def show(fact_id):
    """1 事実を全部表示: 2 言語 × 2 言い回し、greedy 全部、サンプルは出現回数つきで集約。"""
    q = next(x for x in qa if x["fact_id"] == fact_id)
    r = results[fact_id]
    print(f"=== {fact_id}  [{q['type']}]  {q['article']}")
    print(f"  gold       : {q['answer']!r}")
    print(f"  aliases_en : {q.get('aliases_en', [])}")
    print(f"  aliases_ja : {q.get('aliases_ja', [])}")
    print(f"  evidence   : {facts[fact_id].get('evidence', '')}")
    for lang, key in (("en", "q_en"), ("ja", "q_ja")):
        d = r[lang]
        print(f"\n  [{lang}] {d['label']}   PCorrect(T=0)={d['p_correct_greedy']}  "
              f"PCorrect(T>0)={d['p_correct_sample']}  (greedy {d['n_greedy']} / samples {d['n_sample']})")
        for j, qq in enumerate(q[key][:2]):
            print(f"    Q{j}: {qq}")
        for j, a in enumerate(d["greedy"]):
            mark = "✓" if pk.is_correct(a, pk.golds_for(q, lang)) else " "
            print(f"    greedy[{j}] {mark} {a!r}")
        for a, n in collections.Counter(d["samples"]).most_common():
            mark = "✓" if pk.is_correct(a, pk.golds_for(q, lang)) else " "
            print(f"    sample ×{n:<3d} {mark} {a!r}")


show(qa[0]["fact_id"])


In [ ]:
art = df.groupby("article").agg(n=("fact_id", "size"), known_en=("known_en", lambda s: s.isin(["HighlyKnown", "MaybeKnown"]).mean()),
                                known_ja=("known_ja", lambda s: s.isin(["HighlyKnown", "MaybeKnown"]).mean()), inject=("inject", "sum"))
print("articles with the highest share of Known facts (EN):")
display(art.sort_values("known_en", ascending=False).head(12).round(2))
print(f"\narticles with 0 injectable facts: {int((art['inject'] == 0).sum())} / {len(art)}")

In [ ]:
print("EN HighlyKnown（推測可能・常識的な事実の例）:")
display(df[df["known_en"] == "HighlyKnown"][["article", "subject", "relation", "object", "greedy_en"]].head(10))
print("\nUnknown だが棄権していない例（confabulation）:")
conf = df[df["inject"] & (df["wrong_en"] == N_GREEDY)][["article", "relation", "object", "greedy_en", "greedy_ja"]]
display(conf.head(10))

In [ ]:
script_of = pk.script_of   # 文字種判定。§2.6 の「言語違い」カテゴリでも同じ関数を使う

ja = pd.DataFrame([{"fact_id": q["fact_id"], "article": q["article"], "object": q["answer"], "answer": a,
                    "correct": pk.is_correct(a, [q["answer"]] + q.get("aliases_en", []) + q.get("aliases_ja", [])),
                    "script": script_of(a)} for q in qa for a in results[q["fact_id"]]["ja"]["greedy"]])
print("JA greedy answers by script:\n" + ja["script"].value_counts().to_string())
print("\nhan-only, not counted correct（中国語回答 or 別名漏れの候補）:")
display(ja[(ja["script"] == "han-only") & ~ja["correct"]].head(15))

## 7. 保存

`facts_probed.jsonl` は `scripts/probe_known.py` と同じ形式（事実行 + `known_en` / `known_ja` + `probe`）。`probe_summary.json` はベースモデルの数字（論文の Baseline 節用）。

In [ ]:
with open(OUT_FILE, "w", encoding="utf-8") as out:
    for q in qa:
        f = dict(facts[q["fact_id"]]); r = results[q["fact_id"]]
        f["known_en"], f["known_ja"] = r["en"]["label"], r["ja"]["label"]
        f["probe"] = {lang: {k: r[lang][k] for k in ("label", "p_correct_greedy", "p_correct_sample",
                             "f1_greedy_max", "f1_sample_max", "n_greedy", "n_sample",
                             "greedy", "samples", "abstain_greedy")} for lang in ("en", "ja")}
        out.write(json.dumps(f, ensure_ascii=False) + "\n")

summary = {
    "model": model_name, "n_facts": len(df), "n_inject": int(df["inject"].sum()), "nex": N_EX, "samples": N_SAMPLES, "temperature": TEMPERATURE, "top_p": TOP_P,
    "n_greedy_per_lang": N_GREEDY, "n_sample_per_lang": N_GREEDY * N_SAMPLES,
    "labels": {lang: df[f"known_{lang}"].value_counts().to_dict() for lang in ("en", "ja")},
    "cross_en_ja": {e: {j: int(cross.loc[e, j]) for j in LABELS} for e in LABELS},
    "greedy_rates": {lang: rates.loc[lang].round(4).to_dict() for lang in ("en", "ja")},
    "by_type": by_type.to_dict("index"), "by_domain": by_domain.to_dict("index"),
    "control": ctrl_df.to_dict("records") if RUN_CONTROL else None,
    "date": time.strftime("%Y-%m-%d %H:%M"),
}
json.dump(summary, open(SUMMARY_FILE, "w", encoding="utf-8"), indent=1, ensure_ascii=False)
print(f"wrote {len(qa)} facts → {OUT_FILE}\nsummary → {SUMMARY_FILE}")

## 8. 次のステップ

```bash
# 注入セット（両言語 Unknown）の記事だけ言い換えを作る
python scripts/make_paraphrases.py --facts experiment_data/facts_probed.jsonl --unknown-only

# パイロット（100 事実、full FT）→ 目安: EN で 0.6 以上
papermill notebook/experiment_4_finetune.ipynb outputs/nb-pilot-full.ipynb -p METHOD full -p N_FACTS 100 -p RUN_TAG pilot
```

- 注入セットが少なすぎる場合（目安 500 事実未満）: MaybeKnown / WeaklyKnown を「JA のみ Unknown」で緩める案もあるが、転移の解釈が濁るので **まず記事を増やす** 方を優先する
- han-only の回答で正しい表記が `aliases_ja` に無いものは `eval_qa.jsonl` の別名に足してからプローブを再実行する（キャッシュは QA 内容のハッシュ付きなので自動で作り直される）